In [16]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import json
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_gigachat.chat_models import GigaChat
from ddgs import DDGS
import time
import re
from model import model

In [17]:



SITES = [
    "cbr.ru",
    "rbk.ru", 
    "vedomosti.ru",
    "ru.investing.com",
    "ria.ru",
    "lenta.ru",
    "interfax.ru",
    "forbes.ru",
    "finam.ru",
    "frankmedia.ru",
    "banki.ru"
]

KEYWORDS = [
    "валюта", "акции", "облигации", "ставка ЦБ", "ключевая ставка", 
    "курс", "рубль" "доллар", "евро", "фондовый рынок", "инвестиции"
]


DAYS_LIMIT = 30
date_limit = datetime.now() - timedelta(days=DAYS_LIMIT)

In [1]:

# PromptTemplate для GigaChat
prompt_template = PromptTemplate(
    input_variables=["articles"],
    template=("""
        Ты финансовый аналитик.
        Проанализируй список новостных статей за последние 30 дней:
        {articles}
        Отбери те новости, которые наиболее важны для инвесторов по темам: валюта, акции, облигации, ставка ЦБ, ключевая ставка, курс рубля доллара евро, фондовый рынок, инвестиции
        Верни только релевантные новости в формате JSON со следующими ключами: title, summary, date
        Исключи дублирующиеся новости и оставь только самые важные.
        Бери новости только с ключевой информацией, фактами и конкретными значениями.
        Строго - Верни данные в формате JSON.
        Пример вывода:
              [
                {{
                    "title": "Ставка ЦБ за снизилась на 1 процент",
                    "summary": "Анализ",
                    "date": "2025-09-15"
                }},
                {{
                    "title": "Акции компании ** выросли на **",
                    "summary": "Анализ",
                    "date": "2025-09-15"
                }}
              ]
    """)
)

NameError: name 'PromptTemplate' is not defined

In [ ]:
def search_news_by_site(site, keywords, max_results=20):
    results = []
    ddgs = DDGS()
    
    for keyword in keywords[:3]:  # Можно ограничить количество ключевых слов для ускорения
        try:
            query = f"site:{site} {keyword}"
            search_results = ddgs.text(
                query=query, 
                region='ru-ru', 
                safesearch='moderate',
                timelimit='m',  # поиск за последний месяц
                max_results=max_results
            )
            
            for result in search_results:
                if result.get('href') and site in result['href']:
                    results.append({
                        'url': result['href'],
                        'title': result.get('title', ''),
                        'summary': result.get('body', '')
                    })
            
            time.sleep(1)
            
        except Exception as e:
            print(f"Ошибка поиска на {site} по ключевому слову '{keyword}': {e}")
    
    return results

def extract_article_date(url):
    """Извлечение даты публикации статьи"""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Поиск даты в meta-тегах
        date_selectors = [
            'meta[property="article:published_time"]',
            'meta[name="publishdate"]',
            'meta[name="date"]',
            'meta[itemprop="datePublished"]',
            'time[datetime]',
            '.date', '.time', '.published'
        ]
        
        for selector in date_selectors:
            date_element = soup.select_one(selector)
            if date_element:
                date_content = date_element.get('content') or date_element.get('datetime') or date_element.text
                if date_content:
                    # Попытка парсинга различных форматов даты
                    date_patterns = [
                        r'\d{4}-\d{2}-\d{2}',
                        r'\d{2}\.\d{2}\.\d{4}',
                        r'\d{1,2} \w+ \d{4}'
                    ]
                    
                    for pattern in date_patterns:
                        match = re.search(pattern, date_content)
                        if match:
                            try:
                                date_str = match.group()
                                if '-' in date_str:
                                    return datetime.strptime(date_str, '%Y-%m-%d')
                                elif '.' in date_str:
                                    return datetime.strptime(date_str, '%d.%m.%Y')
                            except ValueError:
                                continue
        
        return None
        
    except Exception as e:
        print(f"Ошибка извлечения даты из {url}: {e}")
        return None

def collect_all_news():
    """Сбор всех новостей с указанных сайтов"""
    all_articles = []
    
    print("Начинаем сбор новостей...")
    
    for site in SITES:
        print(f"Ищем новости на {site}...")
        site_articles = search_news_by_site(site, KEYWORDS, max_results=15)
        
        for article in site_articles:
            # Проверяем дату публикации
            pub_date = extract_article_date(article['url'])
            
            if pub_date and pub_date >= date_limit:
                article['date'] = pub_date.strftime('%Y-%m-%d')
                all_articles.append(article)
                safe_title = article['title'].encode('utf-8', errors='replace').decode('utf-8')
                print(f"Найдена актуальная статья: {safe_title[:80]}...")
            elif not pub_date:
                # Если дату не удалось извлечь, добавляем статью
                article['date'] = "Дата неизвестна"
                all_articles.append(article)
        
        time.sleep(2)  # Пауза между сайтами
    
    print(f"Всего собрано статей: {len(all_articles)}")
    return all_articles

def prepare_articles_text(articles):
    text_parts = []
    for i, article in enumerate(articles, 1):
        part = f"""
Статья {i}:
Заголовок: {article['title']}
URL: {article['url']}
Дата: {article['date']}
Описание: {article['summary'][:300]}...
"""
        # Экранируем фигурные скобки
        part = part.replace('{', '{{').replace('}', '}}')
        text_parts.append(part)

    return "\n".join(text_parts)

def analyze_with_gigachat(articles):
    if not articles:
        return []

    print("Отправляем статьи в GigaChat для анализа...")
    text = "\n".join(
        f"Статья {i}: Заголовок: {a['title']} Дата: {a['date']} Описание: {a['summary']}"
        for i, a in enumerate(articles, 1)
    )
    prompt = prompt_template.format(articles=text)

    response = model.invoke(prompt)
    # Получаем тело ответа и причину остановки генерации
    content = getattr(response, "content", "") or ""
    reason = getattr(response, "reason", "")

    if not content:
        print("Пустой ответ GigaChat, причина генерации:", reason)
        return []
    if reason.lower() == "blacklist" or "blacklist" in content.lower():
        print("Ответ заблокирован фильтром (blacklist), пропускаем пакет")
        return []

    # Ищем JSON-массив во всём тексте
    match = re.search(r"\[.*\]", content, re.DOTALL)
    json_str = match.group() if match else content

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print("Ошибка парсинга JSON:", e)
        return []

def save_results(raw_articles, analyzed_articles):
    """Сохранение результатов в JSON файлы"""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Сохраняем исходные статьи
    with open(f'news_raw.json', 'w', encoding='utf-8') as f:
        json.dump(raw_articles, f, ensure_ascii=False, indent=2)
    
    # Сохраняем обработанные статьи
    with open(f'news_analyzed.json', 'w', encoding='utf-8') as f:
        json.dump(analyzed_articles, f, ensure_ascii=False, indent=2)
    
    print(f"Результаты сохранены в файлы:")
    print(f"- news_raw.json")
    print(f"- news_analyzed.json")

def batch_analyze_with_gigachat(articles, batch_size=10):
    analyzed = []
    for idx in range(0, len(articles), batch_size):
        batch = articles[idx : idx + batch_size]
        result = analyze_with_gigachat(batch)
        if result:
            analyzed.extend(result)
            print(f"Пакет {idx//batch_size+1}: добавлено {len(result)} статей")
        else:
            print(f"Пакет {idx//batch_size+1}: нет валидного ответа")
    return analyzed

def main():
    """Основная функция"""
    print("=== Агрегатор финансовых новостей ===")
    
    # Сбор новостей
    raw_articles = collect_all_news()
    
    if not raw_articles:
        print("Новости не найдены.")
        return
    
    # Анализ через GigaChat
    analyzed_articles = batch_analyze_with_gigachat(raw_articles, batch_size=10)
    
    # Сохранение результатов
    save_results(raw_articles, analyzed_articles)
    
    print("=== Готово! ===")
    
    # Показываем краткую статистику
    if isinstance(analyzed_articles, list):
        print(f"Отобрано релевантных новостей: {len(analyzed_articles)}")
        for article in analyzed_articles[:3]:  # Показываем первые 3
            if isinstance(article, dict) and 'title' in article:
                print(f"- {article['title']}")

if __name__ == "__main__":
    main()


=== Агрегатор финансовых новостей ===
Начинаем сбор новостей...
Ищем новости на cbr.ru...
Ищем новости на rbk.ru...


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Ищем новости на vedomosti.ru...
Найдена актуальная статья: Как на снижение ставки ЦБ отреагировали банки, рынки и валюта...
Найдена актуальная статья: Акции Nvidia дешевели из-за нарушения антимонопольного ......
Найдена актуальная статья: Облигации регионов Дальнего Востока и Арктики появятся на ......
Ищем новости на ru.investing.com...
Ищем новости на ria.ru...
Ищем новости на lenta.ru...
Ищем новости на interfax.ru...
Найдена актуальная статья: Рубль утром повышается в паре с юанем на фоне стабильной нефти...
Найдена актуальная статья: Акции Московская Биржа (MOEX) - цена на сегодня, график ......
Найдена актуальная статья: Чукотка, Якутия и Хабаровский край планируют ......
Ищем новости на forbes.ru...
Ищем новости на finam.ru...
Ищем новости на frankmedia.ru...
Ищем новости на banki.ru...
Всего собрано статей: 33
Отправляем статьи в GigaChat для анализа...
Пакет 1: добавлено 5 статей
Отправляем статьи в GigaChat для анализа...
Пакет 2: добавлено 6 статей
Отправляем статьи в GigaC